In [ ]:
import random
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
def read_excel_files(folder_path):

    # List all Excel files in the folder
    all_files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f)) and f.endswith('.xlsx')]
    
    # List to store DataFrames
    dfs_list = {}

    for file in all_files:
        file_path = os.path.join(folder_path, file)
        df = pd.read_excel(file_path, engine='openpyxl')
        
        # Clean up the matrix (if the 'Unnamed: 0' column exists)
        if 'Unnamed: 0' in df.columns:
            df = df.set_index('Unnamed: 0')
            df.index.name = None
        
        file_name = os.path.basename(file_path)
        dfs_list[file_name] = df

    return dfs_list

In [ ]:
predictedMatrixAdjacencyPath = "MatrixAdjacency/Predicted/"
desiredMatrixAdjacencyPath = "MatrixAdjacency/Desired/"
filteredDataPath = "MatrixAdjacency/FilteredData/"

desired_adj_matrices = read_excel_files(desiredMatrixAdjacencyPath)
predicted_adj_matrices = read_excel_files(predictedMatrixAdjacencyPath)
filtered_data = read_excel_files(filteredDataPath)

In [ ]:
desired_specific_filename = 'rectified_matrix_2020-02-28.xlsx'
predicted_specific_filename = 'predicted_matrix_2020-02-28.xlsx'
filtered_specific_filename = 'stock_price_2020-02-20_2020-02-28.xlsx'

specific_desired_matrix = desired_adj_matrices.get(desired_specific_filename, None)
specific_predicted_matrix = predicted_adj_matrices.get(predicted_specific_filename, None)
specific_filtered_data = filtered_data.get(filtered_specific_filename, None)

if specific_predicted_matrix is not None:
    specific_predicted_df = pd.DataFrame(specific_predicted_matrix)
else:
    print("Matrix not found for the specified file name.")

if specific_desired_matrix is not None:
    specific_desired_df = pd.DataFrame(specific_desired_matrix)
else:
    print("Matrix not found for the specified file name.")

if specific_filtered_data is not None:
    specific_filtered_df = pd.DataFrame(specific_filtered_data)
else:
    print("Data not found for the specified file name.")

specific_filtered_df

In [ ]:
# Selected companies grouped by sector
companies = [
    "^GSPC",              # Index
    "SHW", "ECL", "PPG",  # Materials
    "BA", "CAT", "MMM",   # Industrials
    "AMZN", "F", "NKE",   # Consumer Discretionary
    "PG", "KO", "PEP",    # Consumer Staples
    "JNJ", "PFE", "MRK",  # Health Care
    "JPM", "BAC", "WFC",  # Financials
    "AAPL", "MSFT", "INTC", # Information Technology
    "GOOGL", "T", "CMCSA", # Communication Services
    "DUK", "SO", "D",     # Utilities
    "SPG", "AMT", "EQIX", # Real Estate
    "XOM", "CVX", "SLB"   # Energy
]

# Extract submatrices for selected companies
desired_data = specific_desired_df.loc[companies, companies]
predicted_data = specific_predicted_df.loc[companies, companies]

# Display results
# print(desired_data)
print(predicted_data)
print(predicted_data.iloc[0])

In [ ]:
desired_data.to_excel("specific_desired_data_2020-02-28.xlsx")
predicted_data.to_excel("specific_predicted_data_2020-02-28.xlsx")

In [ ]:
# Compute total degree of connections
total_degree_desired = specific_desired_df.loc[companies].sum(axis=1)
total_degree_predicted = specific_predicted_df.loc[companies].sum(axis=1)

# Compute specific degree of connections (within selected companies)
specific_degree_desired = specific_desired_df.loc[companies, companies].sum(axis=1)
specific_degree_predicted = specific_predicted_df.loc[companies, companies].sum(axis=1)

# Display results
print("\nSpecific degree for each company (desired): " + desired_specific_filename)
print(specific_degree_desired)

print("\nSpecific degree for each company (predicted): " + predicted_specific_filename)
print(specific_degree_predicted)

In [ ]:
import networkx as nx

def calculate_centrality(df):
    # Create a graph from the adjacency matrix
    G = nx.from_pandas_adjacency(df)

    # Calculate betweenness centrality
    betweenness = nx.betweenness_centrality(G)

    # Calculate closeness centrality
    closeness = nx.closeness_centrality(G)

    return betweenness, closeness

# For your desired and predicted matrices
betweenness_desired, closeness_desired = calculate_centrality(desired_data)
betweenness_predicted, closeness_predicted = calculate_centrality(predicted_data)

# Print the results
print("Betweenness Centrality (Desired):", betweenness_desired)
print("Closeness Centrality (Desired):", closeness_desired)
print("\nBetweenness Centrality (Predicted):", betweenness_predicted)
print("Closeness Centrality (Predicted):", closeness_predicted)

In [ ]:
def calculate_network_indicators(df):
    # Create a graph from the adjacency matrix
    G = nx.from_pandas_adjacency(df)

    # Calculate network indicators
    density = nx.density(G)
    if nx.is_connected(G):
        diameter = nx.diameter(G)
    else:
        diameter = None  # Diameter is not defined for disconnected graphs
    clustering_coefficient = nx.average_clustering(G)
    degree_centrality = nx.degree_centrality(G)

    return density, diameter, clustering_coefficient, degree_centrality

# For your desired and predicted matrices
density_desired, diameter_desired, clustering_coefficient_desired, degree_centrality_desired = calculate_network_indicators(desired_data)
density_predicted, diameter_predicted, clustering_coefficient_predicted, degree_centrality_predicted = calculate_network_indicators(predicted_data)

# Print the results
print("Network Indicators for Desired Data:")
print("Density:", density_desired)
print("Diameter:", diameter_desired)
print("Clustering Coefficient:", clustering_coefficient_desired)
print("Degree Centrality:", degree_centrality_desired)

print("\nNetwork Indicators for Predicted Data:")
print("Density:", density_predicted)
print("Diameter:", diameter_predicted)
print("Clustering Coefficient:", clustering_coefficient_predicted)
print("Degree Centrality:", degree_centrality_predicted)

In [ ]:
if specific_filtered_df is not None:
    # Filter DataFrame to include only selected companies
    filtered_df = specific_filtered_df[companies]

    # Compute daily changes for each company
    daily_changes = filtered_df.diff()

    # Compute average changes for each company
    average_changes = daily_changes.mean()

    print("Average return change for each stock:")
    print(average_changes)
else:
    print("File not found or DataFrame is empty.")

In [ ]:
# Companies grouped by sector
sectors = {
    "Materials": ["SHW", "ECL", "PPG"],
    "Industrials": ["BA", "CAT", "MMM"],
    "Consumer Discretionary": ["AMZN", "F", "NKE"],
    "Consumer Staples": ["PG", "KO", "PEP"],
    "Health Care": ["JNJ", "PFE", "MRK"],
    "Financials": ["JPM", "BAC", "WFC"],
    "Information Technology": ["AAPL", "MSFT", "INTC"],
    "Communication Services": ["GOOGL", "T", "CMCSA"],
    "Utilities": ["DUK", "SO", "D"],
    "Real Estate": ["SPG", "AMT", "EQIX"],
    "Energy": ["XOM", "CVX", "SLB"]
}

In [ ]:
if specific_filtered_df is not None:
    average_sector_variation = {}

    for sector, companies in sectors.items():
        # Filter DataFrame to include only companies in the sector
        sector_df = specific_filtered_df[companies]

        # Compute daily changes for each company
        daily_changes = sector_df.diff()

        # Compute average variation for the sector
        average_changes = daily_changes.mean().mean()  # Mean of company means

        average_sector_variation[sector] = average_changes

    print("Average return change for each sector:")
    print(average_sector_variation)
else:
    print("File not found or DataFrame is empty.")

In [ ]:
# Companies grouped by sector, including the index
sectors = {
    "Index": ["^GSPC"],
    "Materials": ["SHW", "ECL", "PPG"],
    "Industrials": ["BA", "CAT", "MMM"],
    "Consumer Discretionary": ["AMZN", "F", "NKE"],
    "Consumer Staples": ["PG", "KO", "PEP"],
    "Health Care": ["JNJ", "PFE", "MRK"],
    "Financials": ["JPM", "BAC", "WFC"],
    "Information Technology": ["AAPL", "MSFT", "INTC"],
    "Communication Services": ["GOOGL", "T", "CMCSA"],
    "Utilities": ["DUK", "SO", "D"],
    "Real Estate": ["SPG", "AMT", "EQIX"],
    "Energy": ["XOM", "CVX", "SLB"]
}

In [ ]:
if specific_filtered_df is not None:
    amplitude_per_stock = {}
    average_amplitude_per_sector = {}

    for sector, companies in sectors.items():
        sector_amplitudes = []

        for company in companies:
            # Compute daily changes for the company
            daily_changes = specific_filtered_df[company].diff()

            # Compute amplitude of variation for the company
            amplitude = daily_changes.max() - daily_changes.min()
            amplitude_per_stock[company] = amplitude

            sector_amplitudes.append(amplitude)

        # Compute average amplitude for the sector
        average_amplitude_per_sector[sector] = sum(sector_amplitudes) / len(sector_amplitudes)

    print("Amplitude per stock:")
    print(amplitude_per_stock)

    print("\nAverage amplitude per sector:")
    print(average_amplitude_per_sector)
else:
    print("File not found or DataFrame is empty.")